In [ ]:
# --- make the safety_probes package importable ---
# Preferred: run `pip install -e .` from the repo root once; then this cell is a no-op.
import sys, pathlib
try:
    import safety_probes
except ModuleNotFoundError:
    starts = []
    _f = globals().get("__vsc_ipynb_file__")
    if _f: starts.append(pathlib.Path(_f).resolve())
    starts.append(pathlib.Path.cwd())
    for _s in starts:
        _root = next((p for p in [_s, *_s.parents] if (p / "safety_probes" / "__init__.py").exists()), None)
        if _root:
            sys.path.insert(0, str(_root)); break
    import safety_probes
print("safety_probes:", safety_probes.__file__)

# Cross-behaviour comparison (plan #48)

Compares the behaviours' **steering** results side by side — the cross-behaviour analogues of
[CAA Fig. 5] (dose-response per behaviour) and van der Weij et al. Fig. 7 (simultaneous steering).
Run **after** the behaviour notebooks: §1 reads their persisted artifacts (`exported_probes/*.json`)
and needs **no GPU**; §2 is an EXPERIMENTAL draft that loads the model and the exported bundles.

Design decisions (2026-07-30):
- **MM direction only** — the reference/causal-hypothesis direction, held fixed so the comparison is
  across *behaviours*, not across estimators.
- **OOD eval sets only** — OOD is the claim. ID stays in the behaviour notebooks as the positive
  control (deception ID is NIE-only there, #46, so an ID judge overlay would be one-sided anyway).
- Cross-behaviour **probe similarities deliberately left out** (user decision; cf. plan #41 — the
  behaviours' detection layers differ, so vector comparisons would cross residual bases).
- `BEHAVIOURS` below is the extension point: sycophancy joins by adding its name (and its spec to
  `safety_probes/behaviours.py`).

## 1. Steering dose-response across behaviours (GPU-free)

Three panels, one line per (behaviour, OOD set), MM direction:
- **NIE** — the honest cross-behaviour axis: every behaviour's NIE is in *its own class-gap units*
  (coef +1 = one full gap), so curve heights are comparable in a way raw judge scores are not.
- **Judge behaviour** — the literal [CAA Fig. 5] analogue. Rubrics differ per behaviour, so compare
  *shapes* (slopes, sign response), not absolute levels.
- **Judge coherence** — the guard: any behaviour "win" at collapsed coherence is steering-by-breaking.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from safety_probes import summary

# Crisp inline figures + the shared DPI bump (plan #50/M15).
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200})

BEHAVIOURS = ["deception", "high-stakes"]   # sycophancy joins here (label = its save_* name)
METHOD = "MM"

try:
    full = {b: summary.load_steering_full(b) for b in BEHAVIOURS}
except FileNotFoundError as e:
    raise SystemExit(f"Missing artifact ({e}) - run the behaviour notebooks first; their export "
                     "cells write exported_probes/<behaviour>_steering_full.json.")

try: plt.style.use("seaborn-v0_8-darkgrid")
except OSError: pass
panels = [("nie", "readout NIE (class-gap units)"), ("judge_beh", "judge: behaviour (1-10)"),
          ("judge_coh", "judge: coherence (1-10)")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for b in BEHAVIOURS:
    for label, res in full[b].items():
        if "OOD" not in label:   # ID = positive control; lives in the behaviour notebooks
            continue
        r = res[METHOD]
        for ax, (key, _) in zip(axes, panels):
            y = np.array(r[key], dtype=float)
            if np.all(np.isnan(y)):
                continue
            ax.plot(r["coef"], y, marker="o", label=f"{b}: {label}")
for ax, (key, title) in zip(axes, panels):
    ax.set_title(title)
    ax.set_xlabel("steering coefficient (1.0 = cross class-mean gap)")
    ax.axvline(0, color="grey", lw=0.8, ls="--")
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].legend(fontsize=8)   # legend on the NIE panel: it always has lines (judge panels may be NaN)
fig.suptitle(f"Cross-behaviour steering dose-response - {METHOD} direction, OOD eval sets")
fig.tight_layout()
plt.show()

## 2. Simultaneous steering — EXPERIMENTAL DRAFT

The van der Weij et al. Fig. 7 analogue: **one global coefficient** scales every behaviour's
calibrated MM vectors at once (their layer blocks are disjoint for the current bundles — deception
[15-19], high-stakes [9-13] — so this is exactly their "each vector at its own layer" setting), and
each behaviour's **own readout** measures its shift, in its own class-gap units.

⚠️ `compose_steering` is flagged RESEARCH-GRADE (plan §4.4): whether behaviour directions compose,
cancel, or break the model is an open question no reference paper answers. Treat any output as a
pilot observation until it has been through the full NIE + judge-coherence protocol. The judge/
coherence ("alignment tax") axis is deliberately left out of this draft — it belongs in the
validated version, which becomes the headline figure once the third behaviour (sycophancy) exists.

Flip `RUN_SIMULTANEOUS = True` to run (loads the 8-bit model; forward passes only, no generation).

In [ ]:
RUN_SIMULTANEOUS = False   # <-- flip to run the draft (needs GPU + both exported bundles)

if RUN_SIMULTANEOUS:
    from safety_probes import runtime, datasets, export
    from safety_probes.behaviours import DECEPTION_SPEC, HIGH_STAKES_SPEC
    from safety_probes.steering import compose_steering, readout_pdiff, nie_baseline_gap

    runtime.init()
    dec_mm = export.load_behaviour("deception",   expect_model=runtime.MODEL_NAME, expect_d_model=runtime.D_MODEL)["steering"]["MM"]
    hs_mm  = export.load_behaviour("high_stakes", expect_model=runtime.MODEL_NAME, expect_d_model=runtime.D_MODEL)["steering"]["MM"]

    # Readout sets = each behaviour's OOD steering readouts, rebuilt as in its notebook (§7c).
    dd = datasets.ensure_deception_data()
    items = datasets.load_ai_liar(dd).extra["items"]
    dec_neg = [f"System: {it['normal_instruction']}\nUser: {it['question_prefix']}{it['question']}" for it in items[:27]]
    dec_pos = [f"System: {it['deceive_instruction']}\nUser: {it['question_prefix']}{it['question']}" for it in items[:27]]
    hh = datasets.load_high_stakes_ood()["anthropic_hh_balanced"]
    hh_raw, hh_lab = hh.extra["raw"], hh.labels.tolist()
    hs_neg = [x for x, l in zip(hh_raw, hh_lab) if l == 0][:32]
    hs_pos = [x for x, l in zip(hh_raw, hh_lab) if l == 1][:32]

    # Per-behaviour NIE denominators. nie_baseline_gap warns loudly on a degenerate gap; the 1e-6
    # clamp below only prevents a division blow-up (same numerical guard as evaluation.py).
    gap_d, base_d = nie_baseline_gap(DECEPTION_SPEC,   dec_pos, dec_neg)
    gap_h, base_h = nie_baseline_gap(HIGH_STAKES_SPEC, hs_pos,  hs_neg)
    den_d = gap_d if abs(gap_d) > 1e-6 else 1e-6
    den_h = gap_h if abs(gap_h) > 1e-6 else 1e-6

    GLOBAL_COEFS = list(np.linspace(-2, 2, 9))
    sim = {"coef": [], "deception": [], "high-stakes": []}
    for c in GLOBAL_COEFS:
        hook = None if c == 0 else compose_steering((dec_mm, c), (hs_mm, c))
        pd_d = readout_pdiff(DECEPTION_SPEC,   dec_neg, steer_hook=hook).mean()
        pd_h = readout_pdiff(HIGH_STAKES_SPEC, hs_neg,  steer_hook=hook).mean()
        sim["coef"].append(c)
        sim["deception"].append((pd_d - base_d) / den_d)
        sim["high-stakes"].append((pd_h - base_h) / den_h)
        print(f"coef {c:+.1f}: deception NIE {sim['deception'][-1]:+.3f} | high-stakes NIE {sim['high-stakes'][-1]:+.3f}")

    fig, ax = plt.subplots(figsize=(7, 4.4))
    for b in ("deception", "high-stakes"):
        ax.plot(sim["coef"], sim[b], marker="o", label=b)
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel("global injection coefficient (both behaviours steered together)")
    ax.set_ylabel("NIE (each behaviour's own class-gap units)")
    ax.set_title("Simultaneous steering - PILOT, not validated (plan #48)")
    ax.legend(); fig.tight_layout(); plt.show()
else:
    print("Simultaneous-steering draft skipped (RUN_SIMULTANEOUS = False).")